# MFRC Cleaning

This notebook inspects the raw parquet splits, visualizes text coverage and split counts,
and prepares the current standardized MFRC format used in this repo.


In [ ]:
from __future__ import annotations

from pathlib import Path
from ast import literal_eval
import csv
import hashlib
import json
import re
import shutil

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import pandas as pd
except Exception as exc:
    raise RuntimeError("pandas is required to use this cleaning notebook.") from exc

from IPython.display import display


try:
    import pyarrow.parquet as pq
except Exception:
    pq = None


sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "Data").exists() or (candidate / "data").exists():
            return candidate
    return Path.cwd().resolve()


ROOT = find_project_root()
DATA_ROOT = ROOT / "Data" if (ROOT / "Data").exists() else ROOT / "data"
RAW_DIR = DATA_ROOT / "raw/mfrc"
OUT_DIR = DATA_ROOT / "processed" / "mfrc"
SAVE_OUTPUTS = False

print("Project root:", ROOT)
print("Raw dir:", RAW_DIR)
print("Output dir:", OUT_DIR)
print("SAVE_OUTPUTS:", SAVE_OUTPUTS)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def normalize_for_csv(value):
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


def write_jsonl(path: Path, rows) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_csv(path: Path, rows, fieldnames=None) -> None:
    ensure_dir(path.parent)
    if not rows:
        return
    if fieldnames is None:
        fieldnames = []
        seen = set()
        for row in rows:
            for key in row:
                if key not in seen:
                    seen.add(key)
                    fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: normalize_for_csv(row.get(k)) for k in fieldnames})


def plot_count(series, title: str, top_n: int = 15):
    counts = series.fillna("<missing>").astype(str).value_counts().head(top_n)
    if counts.empty:
        print(f"No values available for {title}")
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(x=counts.index, y=counts.values, ax=ax, color="#4C72B0")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("count")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


def plot_text_length(series, title: str):
    lengths = series.fillna("").astype(str).str.len()
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(lengths, bins=30, ax=ax, color="#55A868")
    ax.set_title(title)
    ax.set_xlabel("characters")
    plt.tight_layout()
    plt.show()


In [ ]:
if pq is None:
    raise RuntimeError("pyarrow is required to read MFRC parquet files.")

frames = []
for path in sorted(RAW_DIR.glob("*.parquet")):
    df = pq.read_table(path).to_pandas()
    df["split"] = path.stem
    df["source_file"] = path.name
    frames.append(df)

raw_df = pd.concat(frames, ignore_index=True)
print("Raw shape:", raw_df.shape)
display(raw_df.head())
display(pd.DataFrame({"column": raw_df.columns, "missing": raw_df.isna().sum().values}).sort_values("missing", ascending=False))


In [ ]:
text_series = raw_df.get("text", pd.Series(dtype=str)).fillna("").astype(str).str.strip()

plot_count(raw_df["split"], "MFRC split distribution")
plot_text_length(text_series, "MFRC text length distribution")

metadata_columns = [c for c in raw_df.columns if c not in {"text", "split", "source_file"}]
unique_counts = pd.Series({col: raw_df[col].nunique(dropna=True) for col in metadata_columns}).sort_values(ascending=False)
display(unique_counts.to_frame("nunique").head(15))

for column in metadata_columns[:4]:
    if raw_df[column].dtype == "object":
        plot_count(raw_df[column].astype(str), f"MFRC top values for {column}")


In [ ]:
metadata_columns = [c for c in raw_df.columns if c not in {"text", "split", "source_file"}]
cleaned_df = pd.DataFrame({
    "text": raw_df["text"].fillna("").astype(str).str.strip(),
    "label": "",
    "dataset": "mfrc",
    "task": "moral_sentiment_multilabel",
    "split": raw_df["split"].astype(str),
    "source_file": raw_df["source_file"].astype(str),
    "metadata": raw_df[metadata_columns].to_dict(orient="records"),
})

print("Cleaned shape:", cleaned_df.shape)
display(cleaned_df.head())
print("MFRC note: this format is prompt-usable, but not directly supervised in the current pipeline because labels remain nested in metadata.")


In [ ]:
if SAVE_OUTPUTS:
    write_jsonl(OUT_DIR / "mfrc.jsonl", cleaned_df.to_dict(orient="records"))
    write_csv(OUT_DIR / "mfrc.csv", cleaned_df.to_dict(orient="records"))
    print("Wrote cleaned MFRC outputs to", OUT_DIR)
else:
    print("Preview only. Set SAVE_OUTPUTS = True and rerun this cell to write cleaned files.")
